# Caliby scoring for decoding-design-bias

Caliby ([Shuai et al., 2025 - ProteinDesignLab](https://www.biorxiv.org/content/10.1101/2025.09.30.679633v4)) is a Potts-model-based inverse folding method. Unlike the other models in this study, it doesn't produce per-residue categorical log-probabilities - it produces **Potts energies** (`U` global, `U_i` per-residue), which are unnormalized MRF energies from a statistical-physics-style design framework.

**Scoring convention:**
- `caliby_U` = global Potts energy (lower = more native-like)
- `caliby_U_per_res` = `U / L` - per-residue mean energy
- Reported as-is (lower = more native), and also negated as `caliby_score` so the sign matches log-prob scores elsewhere (higher = more native)
- **Not directly comparable** to PiFold / ProteinMPNN / ESM3 / TriFlow scores on an absolute scale. Comparable at the species-rank level - same axis as what DCA/EVcouplings have always provided.
- Adds a new row to the variance decomposition: classical statistical-physics scoring vs. neural parametric scoring.

**Config:** default `caliby` model (trained on all PDB monomers with 0.3Å Gaussian noise augmentation). AF-predicted structures match this training regime.

**Runtime on T4/A100:** Potts scoring is a single forward pass - ~0.5-2s per protein. Full run ≈ 1-4h.

## 1. Setup

In [ ]:
!nvidia-smi -L

In [ ]:
import os, subprocess, sys

if not os.path.isfile('CALIBY_READY'):
    print('Installing Caliby (this takes 3-5 min)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'caliby @ git+https://github.com/ProteinDesignLab/caliby.git',
         '--quiet'],
        check=True,
    )
    with open('CALIBY_READY', 'w') as fh:
        fh.write('done')

import torch
assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
print('Caliby installed. GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/main_plus_r2_r3_scored_filterC_v3.csv'

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'
OUTPUT = f'{DRIVE_OUT_DIR}/caliby_scores.csv'
PDB_CACHE = '/content/pdbs'
CLEAN_CACHE = '/content/pdbs_cleaned'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(PDB_CACHE, exist_ok=True)
os.makedirs(CLEAN_CACHE, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)

## 2. Load model

Weights (~100MB) auto-download from HuggingFace on first call.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from caliby import load_model

MODEL_NAME = 'caliby'  # 'caliby' | 'soluble_caliby' | 'soluble_caliby_v1'
model = load_model(MODEL_NAME)
print(f'{MODEL_NAME} loaded on {model.device}')

## 3. PDB fetcher + scoring wrapper

`clean_pdbs` standardizes AFDB structures (strips unsupported residues, fixes blank chain IDs) so they parse cleanly. `model.score` returns `{"U": [...], "U_i": [...], "seq": [...]}` - we unwrap the single-protein case.

In [ ]:
import requests
from caliby import clean_pdbs

AF_URL_TEMPLATES = [
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v3.pdb',
]

def fetch_pdb(entry, cache_dir=PDB_CACHE):
    for t in AF_URL_TEMPLATES:
        url = t.format(uid=entry)
        version = url.rsplit('model_', 1)[-1].replace('.pdb', '')
        local = os.path.join(cache_dir, f'AF-{entry}-F1-model_{version}.pdb')
        if os.path.exists(local):
            return local
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(local, 'wb') as fh:
                    fh.write(r.content)
                return local
        except Exception:
            continue
    return None

def score_protein(pdb_path):
    """Clean the PDB, score it, return dict with U, U_per_res, length, seq."""
    cleaned = clean_pdbs([pdb_path], out_dir=CLEAN_CACHE)
    if not cleaned:
        return None
    results = model.score(cleaned, batch_size=1, num_workers=0)
    if not results['U']:
        return None
    U = float(results['U'][0])
    U_i = results['U_i'][0]
    seq = results['seq'][0]
    L = len(seq)
    return {
        'caliby_U':          U,
        'caliby_U_per_res':  U / L if L else float('nan'),
        'caliby_score':     -U / L if L else float('nan'),  # sign-flipped for higher=better
        'length':            L,
        'seq':               seq,
    }

In [ ]:
#@title PDB-MODE (R3.3) - score on experimental PDB chains instead of AlphaFold
#@markdown Toggle ON to score the experimental-structure subset. First upload
#@markdown **pdb_scoring_inputs.csv** and unzip **pdb_chain_structs.zip** to
#@markdown `/content/`. Run this cell AFTER the config/fetch_pdb cell and BEFORE
#@markdown the validation/scoring cells. Leave OFF to use AlphaFold (default).
PDB_MODE = False  #@param {type:"boolean"}
if PDB_MODE:
    import os, pandas as pd
    DATASET = "/content/pdb_scoring_inputs.csv"   # Entry, pdb_id, pdb_chain(=A), sequence(=chain), chain_pdb_path
    assert os.path.exists(DATASET), "Upload pdb_scoring_inputs.csv to /content/"
    _CHAINDIR = "/content/pdb_chain_structs"
    _pdb_df = pd.read_csv(DATASET)
    _pmap = {r.Entry: os.path.join(_CHAINDIR, os.path.basename(str(r.chain_pdb_path)))
             for r in _pdb_df.itertuples()}
    def fetch_pdb(entry, *args, **kwargs):   # override: local single-chain experimental PDB
        p = _pmap.get(entry)
        return p if (p and os.path.exists(p)) else None
    # write/resume from a SEPARATE file so the AlphaFold run's checkpoint isn't
    # reused (otherwise resume sees the AF rows as 'already scored' -> remaining 0)
    try:
        OUTPUT = os.path.splitext(OUTPUT)[0] + "_pdb.csv"
        print("OUTPUT ->", OUTPUT)
    except NameError:
        print("WARNING: OUTPUT not defined yet - run the config cell ABOVE this one first.")
    print(f"PDB-MODE ON - {len(_pmap)} experimental-structure inputs; DATASET -> {DATASET}")
    print("'sequence' is the resolved PDB chain; structures are single-chain (chain 'A').")
    print("Scores cover the resolved region; compare to AF2 scores per-residue (R3.3).")
else:
    print("PDB-MODE OFF - using AlphaFold structures (default).")


## 4. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    t0 = time.time(); pdb = fetch_pdb(entry); t_dl = time.time() - t0
    if pdb is None:
        print(entry, 'no PDB'); continue
    t0 = time.time(); res = score_protein(pdb); t_sc = time.time() - t0
    if res is None:
        print(entry, 'scoring failed'); continue
    print(f'{entry} L={res["length"]:4d} '
          f'U={res["caliby_U"]:.2f} U/L={res["caliby_U_per_res"]:.4f}  '
          f'dl={t_dl:.1f}s score_t={t_sc:.1f}s')

## 5. Full run with resume

Sort by sequence length so any memory behavior or internal caching is monotonic. Flushes after every protein - a disconnect loses only the in-flight one.

In [ ]:
from tqdm.auto import tqdm

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = [r for r in rows if r['Entry'] not in already]
todo.sort(key=lambda r: len(r.get('sequence', '')))
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
with open(OUTPUT, open_mode, newline='') as out:
    w = csv.writer(out)
    if open_mode == 'w':
        w.writerow([
            'Entry', 'species', 'domain',
            'caliby_score', 'caliby_U', 'caliby_U_per_res',
            'scored_length', 'dataset_length',
        ])

    counts = {'ok': 0, 'missing_pdb': 0, 'error': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='caliby'):
        entry = r['Entry']
        pdb = fetch_pdb(entry)
        if pdb is None:
            counts['missing_pdb'] += 1; continue
        try:
            res = score_protein(pdb)
        except Exception as exc:
            print(f'[{entry}] {exc}')
            counts['error'] += 1; continue
        if res is None:
            counts['error'] += 1; continue
        w.writerow([
            entry, r.get('species',''), r.get('domain',''),
            f"{res['caliby_score']:.6f}", f"{res['caliby_U']:.6f}", f"{res['caliby_U_per_res']:.6f}",
            res['length'], len(r.get('sequence','')),
        ])
        out.flush()
        counts['ok'] += 1
        if counts['ok'] % 50 == 0:
            torch.cuda.empty_cache()

print('done in', round(time.time() - t_start, 1), 's', counts)

## 6. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
df[['caliby_score','caliby_U','caliby_U_per_res']].describe()